# Benchmarks

## Initialize

In [4]:
%load_ext autoreload
%autoreload 2

import os
import math
import pathlib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import clear_output

import warnings
from lifelines.utils import CensoringType
from lifelines.utils import concordance_index

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
node = !hostname
if "sc" in node[0]:
    base_path = "/sc-projects/sc-proj-ukb-cvd"
else: 
    base_path = "/data/analysis/ag-reils/ag-reils-shared/cardioRS"
print(base_path)

project_label = "22_medical_records"
project_path = f"{base_path}/results/projects/{project_label}"
figure_path = f"{project_path}/figures"
output_path = f"{project_path}/data"

pathlib.Path(figure_path).mkdir(parents=True, exist_ok=True)
pathlib.Path(output_path).mkdir(parents=True, exist_ok=True)

experiment = 220413
experiment_path = f"{output_path}/{experiment}"
pathlib.Path(experiment_path).mkdir(parents=True, exist_ok=True)

/sc-projects/sc-proj-ukb-cvd


In [20]:
endpoints = sorted(pd.read_csv(f"{experiment_path}/endpoints.txt", header=None)[0].to_list())

In [8]:
endpoint_defs = pd.read_feather(f"{output_path}/phecode_defs_220306.feather").query("endpoint==@endpoints").sort_values("endpoint").set_index("endpoint")

In [9]:
data_covariates = pd.read_feather("/sc-projects/sc-proj-ukb-cvd/data/2_datasets_pre/211110_anewbeginning/baseline_covariates_211209.feather")[["eid", "sex_f31_0_0"]].set_index("eid")

In [10]:
data_outcomes = pd.read_feather(f"{output_path}/baseline_outcomes_long_220412.feather").set_index("eid")

In [11]:
data_all = data_outcomes.merge(data_covariates, left_index=True, right_index=True, how="left").reset_index(drop=False).set_index("endpoint")

In [12]:
data_all.head()

,eid,prev,event,time,sex_f31_0_0
endpoint,,,,,
OMOP_4306655,1000018,False,False,11.866089,Female
phecode_001,1000018,False,False,11.866089,Female
phecode_002,1000018,False,False,11.866089,Female
phecode_002-1,1000018,False,False,11.866089,Female
phecode_003,1000018,False,False,11.866089,Female


In [13]:
data_dict = {e: df.reset_index(drop=True).set_index("eid") for e, df in data_all.groupby('endpoint')}

In [14]:
endpoint_defs.sex.unique()

array(['Both', 'Female', 'Male'], dtype=object)

In [15]:
def get_eligable_eids(data_dict, endpoint):

    data_temp = data_dict[endpoint]
    eligibility = endpoint_defs.loc[endpoint]["sex"]
    
    if eligibility == "Both": 
        eids_incl = data_temp.copy().query(f"prev==0").index.to_list()
    else:
        eids_incl = data_temp.copy().query(f"prev==0&sex_f31_0_0==@eligibility").index.to_list()
        
    return {"endpoint": endpoint, 
            "n_eids": len(eids_incl), 
            "eid_list": eids_incl}

In [16]:
d_list = [get_eligable_eids(data_dict, endpoint) for endpoint in tqdm(endpoints)]
eid_df = pd.DataFrame.from_dict(d_list)

  0%|          | 0/1684 [00:00<?, ?it/s]

In [ ]:
#eid_df.set_index("endpoint")["eid_list"].to_dict()

In [17]:
eid_df.to_feather(f"{output_path}/eligable_eids_220414.feather")

In [18]:
eid_df_long = eid_df[["endpoint", "eid_list"]].explode("eid_list").reset_index(drop=True)
eid_df_long.columns = ["endpoint", "eid"]
eid_df_long["endpoint"] = eid_df_long["endpoint"].astype("category")
eid_df_long["eid"] = eid_df_long["eid"].astype("category")

In [19]:
eid_df_long.to_feather(f"{output_path}/eligable_eids_long_220414.feather")